In [7]:
import pandas as pd 
from pydantic import BaseModel, Field 
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Chunking

In [8]:
df_corpus = pd.read_parquet("C:\\Users\\david\\Documents\\Projects\\crypto_chatbot_rag\\data\\files\\sec_corpus_table_refs_test.parquet")    

In [9]:
## CONSTANTS
MAX_TOKENS = 500
OVERLAP_TOKENS = 75
ENCODING_NAME = "cl100k_base"
# Define the encoding for the text splitter
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name=ENCODING_NAME,
    chunk_size=MAX_TOKENS,       # Maximum tokens per chunk
    chunk_overlap=OVERLAP_TOKENS,     # Repeated tokens between adjacent chunks
    separators=["\n\n", "\n", ". ", " ", ""],
)


In [23]:
def chunk_document(document):
    chunk_records = []

    for section in document["sections"]:
        # 1. Create narrative text chunks
        section_text = section.get("text", "").strip()

        if section_text:
            text_chunks = splitter.split_text(section_text)

            for chunk_index, chunk_text in enumerate(text_chunks):
                chunk_records.append(
                    {
                        "ticker": document["ticker"],
                        "company_name": document["company_name"],
                        "form_type": document["form_type"],
                        "filing_date": document["filing_date"],
                        "period_end": document["period_end"],
                        "source_url": document["source_url"],
                        "accession_number": document["accession_number"],
                        "section_id": section["id"],
                        "chunk_title": section["title"],
                        "chunk_type": "text",
                        "table_id": None,
                        "chunk_index": chunk_index,
                        "chunk_id": (
                            f"{document['accession_number']}"
                            f"*section-{section['id']}"
                            f"*text-{chunk_index:04d}"
                        ),
                        "chunk_text": chunk_text,
                    }
                )

        # 2. Create one separate chunk for each table
        for table in section.get("tables", []):
            table_id = table["table_id"]
            table_text = table["markdown"]

            chunk_records.append(
                {
                    "ticker": document["ticker"],
                    "company_name": document["company_name"],
                    "form_type": document["form_type"],
                    "filing_date": document["filing_date"],
                    "period_end": document["period_end"],
                    "source_url": document["source_url"],
                    "accession_number": document["accession_number"],
                    "section_id": section["id"],
                    "chunk_title": section["title"],
                    "chunk_type": "table",
                    "table_id": table_id,
                    "chunk_index": 0,
                    "chunk_id": (
                        f"{document['accession_number']}"
                        f"*section-{section['id']}"
                        f"*{table_id}"
                    ),
                    "chunk_text": table_text,
                }
            )

    return chunk_records
        

In [24]:
def get_chunks_from_corpus(df_corpus):
    chunks_final = []
    for _, row in df_corpus.iterrows():
        document_chunks = chunk_document(row)
        chunks_final.extend(document_chunks)
    return chunks_final
    

In [25]:
test = get_chunks_from_corpus(df_corpus)

In [26]:
test_df = pd.DataFrame(test)
test_df

,ticker,company_name,form_type,filing_date,period_end,source_url,accession_number,section_id,chunk_title,chunk_type,table_id,chunk_index,chunk_id,chunk_text
0,FETH,Fidelity® Ethereum Fund,10-K,2026-02-25,2025-12-31,https://www.sec.gov/Archives/edgar/data/200004...,0001193125-26-071486,1,Business,text,None,0,0001193125-26-071486*section-1*text-0000,Summary\nFidelity Ethereum Fund (the “Trust”) ...
1,FETH,Fidelity® Ethereum Fund,10-K,2026-02-25,2025-12-31,https://www.sec.gov/Archives/edgar/data/200004...,0001193125-26-071486,1,Business,text,None,1,0001193125-26-071486*section-1*text-0001,The Trust’s inception of operation was July 23...
2,FETH,Fidelity® Ethereum Fund,10-K,2026-02-25,2025-12-31,https://www.sec.gov/Archives/edgar/data/200004...,0001193125-26-071486,1,Business,text,None,2,0001193125-26-071486*section-1*text-0002,The Trust provides exposure to the value of et...
3,FETH,Fidelity® Ethereum Fund,10-K,2026-02-25,2025-12-31,https://www.sec.gov/Archives/edgar/data/200004...,0001193125-26-071486,1,Business,text,None,3,0001193125-26-071486*section-1*text-0003,The Ethereum network allows users to write and...
4,FETH,Fidelity® Ethereum Fund,10-K,2026-02-25,2025-12-31,https://www.sec.gov/Archives/edgar/data/200004...,0001193125-26-071486,1,Business,text,None,4,0001193125-26-071486*section-1*text-0004,History of Ethereum\nThe Ethereum network was ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234,FETH,Fidelity® Ethereum Fund,10-K,2026-02-25,2025-12-31,https://www.sec.gov/Archives/edgar/data/200004...,0001193125-26-071486,14,Principal Accounting Fees and Services,table,table-0004,0,0001193125-26-071486*section-14*table-0004,"| | Year ended December 31, 2025 | | | | ..."
235,FETH,Fidelity® Ethereum Fund,10-K,2026-02-25,2025-12-31,https://www.sec.gov/Archives/edgar/data/200004...,0001193125-26-071486,15,"Exhibits, Financial Statement Schedules",text,None,0,0001193125-26-071486*section-15*text-0000,(1) For a list of the financial statements inc...
236,FETH,Fidelity® Ethereum Fund,10-K,2026-02-25,2025-12-31,https://www.sec.gov/Archives/edgar/data/200004...,0001193125-26-071486,15,"Exhibits, Financial Statement Schedules",table,table-0003,0,0001193125-26-071486*section-15*table-0003,Exhibit Number | | Description\n3.1** | | Ce...
237,FETH,Fidelity® Ethereum Fund,10-K,2026-02-25,2025-12-31,https://www.sec.gov/Archives/edgar/data/200004...,0001193125-26-071486,15,"Exhibits, Financial Statement Schedules",table,table-0002,0,0001193125-26-071486*section-15*table-0002,Exhibit Number | | Description\n32.2* | | Ce...
